# Exame B – Previsão da Demanda de Bicicletas Compartilhadas em Seul

## Resumo Inicial e Objetivos do Trabalho

O objetivo deste projeto é analisar o conjunto de dados sobre o sistema público de bicicletas compartilhadas de Seul, na Coreia do Sul, cobrindo o período de dezembro de 2017 a novembro de 2018. O desafio central é compreender os fatores que impulsionam ou reduzem a demanda por bicicletas e construir um modelo preditivo capaz de estimar a quantidade horária de aluguéis (`Rented_Bike_Count`) a partir de dados meteorológicos e do calendário.

### Etapas do Trabalho
- Etapa 1 – Análise Exploratória de Dados (EDA): conhecer o comportamento individual das variáveis, verificar a qualidade dos dados, analisar a sazonalidade e avaliar como fatores de tempo e clima influenciam o volume de aluguéis.
- Etapa 2 – Testes de Hipótese: validar relações estatísticas por meio de testes formais com verificação de suposições.
- Etapa 3 – Modelagem Preditiva: construir um modelo de regressão linear múltipla, realizar diagnóstico de resíduos e preparar a função de previsão para dados de teste cego.

Link do repositório da disciplina: https://github.com/glaucogoncalves/p-e/tree/main/exames/exame-b-bike-sharing


## Seção 1 – Configuração do Ambiente e Reprodutibilidade

Nesta seção inicial, carregamos as bibliotecas necessárias para manipulação, visualização e modelagem. Para assegurar a reprodutibilidade dos resultados, definimos a semente pseudoaleatória padrão (`random_state = 2025`) e exibimos as versões das bibliotecas instaladas.


In [ ]:
# Objetivo: Importar as bibliotecas essenciais, configurar a semente e verificar versões
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

RANDOM_STATE = 2025
print(f"Versões em uso:")
print(f" - pandas:       {pd.__version__}")
print(f" - numpy:        {np.__version__}")
print(f" - scikit-learn: {sklearn.__version__}")

## Seção 2 – Dicionário de Dados e Visão Geral

O dataset reúne registros horários do sistema público de bicicletas de Seul com informações de clima e calendário.

- Período coberto: 01/12/2017 a 30/11/2018 (7.008 registros horários).
- Total de variáveis: 14 colunas (1 alvo, 2 temporais/identificação, 8 numéricas meteorológicas e 3 categóricas de calendário).

### Variáveis de Tempo e Identificação
| Nome da Variável | Descrição | Tipo |
| :--- | :--- | :--- |
| `Date` | Data do registro (`DD/MM/AAAA`). | Data (Texto) |
| `Hour` | Hora do dia a que o registro se refere (0 a 23). | Numérica (Discreta) |

A coluna `Date` não deve ser aplicada diretamente na modelagem, mas serve de base para derivação de variáveis temporais e ordenação cronológica.

### Variável Dependente (Alvo)
| Nome da Variável | Descrição | Faixa / Resumo | Tipo |
| :--- | :--- | :--- | :--- |
| `Rented_Bike_Count` | Quantidade de bicicletas alugadas naquela hora. | Min: 0, Max: 3.556, Média: 704,97, Mediana: 504,5 | Numérica (Discreta) |

### Variáveis Preditoras Meteorológicas
| Nome da Variável | Descrição | Unidade | Tipo |
| :--- | :--- | :--- | :--- |
| `Temperature` | Temperatura do ar (-17,8 a 39,4 °C, média 12,91). | °C | Numérica Contínua |
| `Humidity` | Umidade relativa do ar (0 a 98%, média 58,27). | % | Numérica Contínua |
| `Wind_Speed` | Velocidade do vento (0 a 7,4 m/s, média 1,73). | m/s | Numérica Contínua |
| `Visibility` | Visibilidade horizontal (27 a 2.000, média 1.434,96). | 10 m | Numérica Discreta |
| `Dew_Point_Temperature` | Temperatura do ponto de orvalho (-30,6 a 26,8 °C). | °C | Numérica Contínua |
| `Solar_Radiation` | Radiação solar incidente (0 a 3,52 MJ/m²). | MJ/m² | Numérica Contínua |
| `Rainfall` | Volume de chuva na hora (0 a 29,5 mm). | mm | Numérica Contínua |
| `Snowfall` | Volume de neve na hora (0 a 8,8 cm). | cm | Numérica Contínua |

### Variáveis Preditoras de Calendário
| Nome da Variável | Descrição | Categorias e Contagens | Tipo |
| :--- | :--- | :--- | :--- |
| `Seasons` | Estação do ano. | `Spring` (1.766), `Summer` (1.767), `Autumn` (1.747), `Winter` (1.728) | Categórica (Nominal) |
| `Holiday` | Indicação de feriado. | `No Holiday` (6.660), `Holiday` (348) | Categórica (Binária) |
| `Functioning_Day` | Funcionamento do sistema no dia. | `Yes` (6.774), `No` (234) | Categórica (Binária) |


## Seção 3 – Carga e Inspeção Inicial dos Dados

O primeiro passo é carregar o arquivo `bike_sharing.csv` e verificar os tipos primitivos das colunas, presença de valores nulos e a estrutura inicial do DataFrame.


In [ ]:
# Objetivo: Carregar o conjunto de dados oficial a partir do repositório da disciplina
df = pd.read_csv("https://raw.githubusercontent.com/glaucogoncalves/p-e/refs/heads/main/exames/exame-b-bike-sharing/bike_sharing.csv")

### Verificação de Amostras e Linhas Duplicadas

Pergunta da análise: existem registros repetidos ou medições horárias duplicadas para o mesmo momento temporal?


In [ ]:
df

In [ ]:
# Objetivo: Inspecionar tipos de dados, contagem de registros não nulos e uso de memória
df.info()

In [ ]:
# Objetivo: Verificar a presença de registros duplicados no conjunto de dados
duplicados_linhas = df.duplicated().sum()
duplicados_chave = df.duplicated(subset=['Date', 'Hour']).sum()

print(f"Linhas inteiramente duplicadas: {duplicados_linhas}")
print(f"Linhas duplicadas pela chave temporal (Date, Hour): {duplicados_chave}")

### O que observamos na verificação de duplicatas

- O dataset não possui nenhuma linha inteiramente duplicada (`0` registros).
- A combinação de `Date` e `Hour`, que define o momento exato de cada observação, também é estritamente única ao longo de todas as 7.008 linhas.
- A integridade dos dados está preservada sem necessidade de remoção de amostras duplicadas nessa etapa.


### Conversão e Ordenação da Variável `Date`

In [ ]:
# Objetivo: Converter Date para datetime e reordenar os dados cronologicamente
df['Date_Parsed'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df = df.sort_values(by="Date_Parsed").reset_index(drop=True)
df[['Date', 'Date_Parsed', 'Hour', 'Rented_Bike_Count']].head()

### Justificativa 

A coluna `Date` vem originalmente tipada como texto (`object`) no formato `DD/MM/AAAA`. Trabalhar com ela nesse formato traz dificuldades práticas:
- Impede a ordenação temporal natural, já que uma ordenação por texto compara caracteres alfabeticamente e não datas cronológicas.
- Inviabiliza a visualização consistente da série temporal dia a dia ao longo dos meses.
- Dificulta a extração de informações derivadas como dia da semana.

Por esse motivo, criamos a coluna `Date_Parsed` convertida para `datetime` com `pd.to_datetime` e reordenamos o dataset cronologicamente do primeiro ao último dia de medição.


## Seção 4 – Análise Exploratória: Fatores de Calendário e Sazonalidade

Ao analisar a demanda de bicicletas compartilhadas, uma pergunta natural é: por que uma pessoa decide ou não alugar uma bicicleta em determinada ocasião?

Dois fatores aparecem como pontos de partida: o clima e o período do dia ou do ano. Iniciamos a investigação pelos fatores de calendário e sazonalidade para responder se o calendário por si só explica os momentos em que a demanda sobe ou cai drasticamente.


In [ ]:
# Objetivo: Gerar medidas resumo das variáveis numéricas do conjunto de dados
df.describe()

### 4.1 Sazonalidade e Comportamento por Estações do Ano (`Seasons`)

Pergunta da análise: como a mudança de estação influencia o volume acumulado de aluguéis e qual é a tendência da demanda ao longo do tempo?

Para manter consistência visual nos gráficos temporais, definimos uma paleta de cores fixa por estação.


In [ ]:
# Objetivo: Definir uma paleta de cores padronizada para as quatro estações
fixed_season_palette = {
    'Summer': '#440154',  # Roxo escuro
    'Autumn': '#31688e',  # Azul/Verde escuro
    'Spring': '#21918c',  # Verde azulado
    'Winter': '#7ad151'   # Verde claro
}

In [ ]:
# Objetivo: Visualizar o total acumulado de bicicletas alugadas por estação do ano
bikes_by_season = df.groupby('Seasons')['Rented_Bike_Count'].sum().reset_index()
bikes_by_season = bikes_by_season.sort_values(by='Rented_Bike_Count', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(
    x='Seasons',
    y='Rented_Bike_Count',
    data=bikes_by_season,
    palette='viridis',
    hue='Seasons',
    hue_order=['Summer', 'Autumn', 'Spring', 'Winter'],
    legend=True
)

plt.title('Total de Bicicletas Alugadas por Estação do Ano', fontsize=14, fontweight='bold')
plt.xlabel('Estação (Seasons)', fontsize=12)
plt.ylabel('Total de Bicicletas Alugadas', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Objetivo: Exibir a série temporal diária completa de aluguéis, colorida pela estação do ano
bikes_daily_season = df.groupby(['Date_Parsed', 'Seasons'])['Rented_Bike_Count'].sum().reset_index()
bikes_daily_season = bikes_daily_season.sort_values(by='Date_Parsed')

plt.figure(figsize=(15, 6))
sns.barplot(
    x='Date_Parsed',
    y='Rented_Bike_Count',
    hue='Seasons',
    hue_order=['Summer', 'Autumn', 'Spring', 'Winter'],
    data=bikes_daily_season,
    dodge=False,
    palette=fixed_season_palette
)

plt.title('Total de Bicicletas Alugadas por Dia (Série Temporal Colorida por Estação)', fontsize=14, fontweight='bold')
plt.xlabel('Data (Ordenada Cronologicamente)', fontsize=12)
plt.ylabel('Total de Bicicletas Alugadas', fontsize=12)
plt.xticks([])  # Oculta rótulos individuais de cada dia para evitar sobreposição
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### O que observamos na análise de sazonalidade

Ao analisar os gráficos de aluguel por estação e ao longo do ano, destacam-se os seguintes pontos:
- O volume de aluguéis no inverno (`Winter`) é muito menor do que nas demais estações. Andar de bicicleta no frio rigoroso de Seul é desfavorável.
- Na série cronológica diária, observa-se uma tendência de crescimento contínuo após o término do inverno, com picos elevados no verão e no outono.
- Apesar do crescimento nas estações mais quentes, chamam a atenção várias quedas abruptas de aluguéis em dias pontuais, atingindo níveis tão baixos quanto no inverno.

A questão seguinte passa a ser: o que acontece nesses dias específicos de queda? Seriam feriados, finais de semana ou alguma paralisação do serviço?


### 4.2 Investigação de Feriados e Datas Comemorativas (`Holiday`)

Pergunta da análise: as quedas acentuadas observadas na série temporal coincidem com feriados nacionais?

Para investigar esse comportamento, realizamos uma consulta externa ao calendário de feriados de Seul durante o período da coleta (dezembro/2017 a novembro/2018):

| Data | Nome do Feriado / Celebração |
| :--- | :--- |
| `25/12/2017` | Natal |
| `01/01/2018` | Ano-Novo |
| `15/02/2018` | Véspera de Seollal |
| `16/02/2018` | Seollal / Ano-Novo Lunar |
| `17/02/2018` | Dia seguinte ao Seollal |
| `01/03/2018` | Dia do Movimento de Independência |
| `05/05/2018` | Dia das Crianças |
| `07/05/2018` | Feriado substituto do Dia das Crianças |
| `22/05/2018` | Aniversário de Buda |
| `06/06/2018` | Dia Memorial |
| `15/08/2018` | Dia da Libertação |
| `23/09/2018` | Véspera de Chuseok |
| `24/09/2018` | Chuseok / Ação de Graças |
| `25/09/2018` | Dia seguinte ao Chuseok |
| `26/09/2018` | Feriado substituto do Chuseok |
| `03/10/2018` | Dia da Fundação Nacional |
| `09/10/2018` | Dia do Hangeul |

Fonte de referência do calendário sul-coreano: https://time.is/calendar/2018/South_Korea


In [ ]:
# Objetivo: Comparar o volume agregado de aluguéis entre feriados e dias normais
bikes_holiday_summary = df.groupby('Holiday')['Rented_Bike_Count'].sum().reset_index()

plt.figure(figsize=(8, 5))
sns.barplot(
    x='Holiday',
    y='Rented_Bike_Count',
    data=bikes_holiday_summary,
    palette='viridis',
    hue='Holiday',
    legend=False
)
plt.title('Total de Bicicletas Alugadas: Feriado vs. Dia Comum', fontsize=12, fontweight='bold')
plt.xlabel('Tipo de Dia', fontsize=10)
plt.ylabel('Total de Aluguéis', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Objetivo: Analisar o volume de aluguéis em cada um dos feriados específicos mapeados em Seul
feriados_mapeados = {
    '25/12/2017': 'Natal',
    '01/01/2018': 'Ano-Novo',
    '15/02/2018': 'Véspera de Seollal',
    '16/02/2018': 'Seollal / Ano-Novo Lunar',
    '17/02/2018': 'Dia seguinte ao Seollal',
    '01/03/2018': 'Dia do Movimento de Independência',
    '05/05/2018': 'Dia das Crianças',
    '07/05/2018': 'Feriado substituto do Dia das Crianças',
    '22/05/2018': 'Aniversário de Buda',
    '06/06/2018': 'Dia Memorial',
    '15/08/2018': 'Dia da Libertação',
    '23/09/2018': 'Véspera de Chuseok',
    '24/09/2018': 'Chuseok / Ação de Graças',
    '25/09/2018': 'Dia seguinte ao Chuseok',
    '26/09/2018': 'Feriado substituto do Chuseok',
    '03/10/2018': 'Dia da Fundação Nacional',
    '09/10/2018': 'Dia do Hangeul'
}

daily_rentals = df.groupby('Date')['Rented_Bike_Count'].sum().reset_index()
daily_rentals_holidays = daily_rentals[daily_rentals['Date'].isin(feriados_mapeados.keys())].copy()
daily_rentals_holidays['Nome_Feriado'] = daily_rentals_holidays['Date'].map(feriados_mapeados)
daily_rentals_holidays = daily_rentals_holidays.sort_values(by='Rented_Bike_Count', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(
    x='Rented_Bike_Count',
    y='Nome_Feriado',
    data=daily_rentals_holidays,
    palette='viridis',
    hue='Nome_Feriado',
    legend=False
)
plt.title('Total de Bicicletas Alugadas em Cada Feriado de Seul', fontsize=14, fontweight='bold')
plt.xlabel('Total de Aluguéis no Dia', fontsize=12)
plt.ylabel('Feriado', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### O que observamos na análise de feriados

A comparação entre feriados e dias comuns gerou observações relevantes:
- O total geral de aluguéis em feriados é muito menor simplesmente porque existem poucos feriados no ano (348 horas em feriado contra 6.660 normais no dataset).
- Quando olhamos os feriados individualmente, muitos deles registram grande quantidade de aluguéis, especialmente nos meses mais quentes (como no Dia da Libertação, Dia Memorial e Dia das Crianças).
- Portanto, o fato de ser feriado não reduz a demanda necessariamente e não explica os dias de queda drástica observados na série temporal.

Isso nos leva a testar a próxima hipótese: será que o tipo de dia (dia útil versus final de semana) explica essas variações?


### 4.3 Investigação de Dias Úteis vs. Finais de Semana (`Day_Type`)

Pergunta da análise: o padrão de uso em dias de semana é diferente do final de semana, a ponto de explicar quedas bruscas de aluguel?

### Justificativa – Criação das Variáveis de Tipo de Dia
A partir de `Date_Parsed`, derivamos `Day_of_Week` (0 = Segunda, ..., 6 = Domingo) e agrupamos em `Day_Type` (`Weekday` para dias úteis e `Weekend` para finais de semana). A intenção é verificar se o propósito de locomoção (deslocamento para trabalho/estudo versus lazer) gera um comportamento de demanda muito diferente.


In [ ]:
# Objetivo: Criar features de dia da semana e classificar em dia útil (Weekday) ou final de semana (Weekend)
df['Day_of_Week'] = df['Date_Parsed'].dt.dayofweek
df['Day_Type'] = df['Day_of_Week'].apply(lambda x: 'Weekend' if x >= 5 else 'Weekday')

print("Distribuição dos registros horários por Tipo de Dia:")
print(df['Day_Type'].value_counts())
print("\nAmostra das variáveis temporais criadas:")
df[['Date_Parsed', 'Day_of_Week', 'Day_Type']].drop_duplicates().sort_values(by='Date_Parsed').head(7)

In [ ]:
# Objetivo: Comparar o total acumulado de aluguéis entre dias de semana e finais de semana
total_by_day_type = df.groupby('Day_Type')['Rented_Bike_Count'].sum().reset_index()
total_by_day_type['Total_K'] = total_by_day_type['Rented_Bike_Count'] / 1000

plt.figure(figsize=(8, 5))
sns.barplot(
    x='Day_Type',
    y='Total_K',
    data=total_by_day_type,
    palette='viridis',
    hue='Day_Type',
    legend=False
)
plt.title('Comparativo Geral: Total de Aluguéis em Dias de Semana vs. Finais de Semana', fontsize=12, fontweight='bold')
plt.xlabel('Tipo de Dia', fontsize=10)
plt.ylabel('Total de Aluguéis (em Milhares - 10³)', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Objetivo: Comparar as séries temporais de dias de semana e finais de semana separadamente
daily_summary = df.groupby(['Date_Parsed', 'Day_Type', 'Seasons'])['Rented_Bike_Count'].sum().reset_index()

weekdays_data = daily_summary[daily_summary['Day_Type'] == 'Weekday'].sort_values(by='Date_Parsed')
weekends_data = daily_summary[daily_summary['Day_Type'] == 'Weekend'].sort_values(by='Date_Parsed')

plt.figure(figsize=(15, 5))
sns.barplot(
    x='Date_Parsed',
    y='Rented_Bike_Count',
    hue='Seasons',
    hue_order=['Summer', 'Autumn', 'Spring', 'Winter'],
    data=weekdays_data,
    dodge=False,
    palette=fixed_season_palette
)
plt.title('Aluguel Diário de Bicicletas – Dias de Semana (Colorido por Estação)', fontsize=14, fontweight='bold')
plt.xlabel('Data', fontsize=12)
plt.ylabel('Total de Aluguéis', fontsize=12)
plt.xticks([])
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

plt.figure(figsize=(15, 5))
sns.barplot(
    x='Date_Parsed',
    y='Rented_Bike_Count',
    hue='Seasons',
    hue_order=['Summer', 'Autumn', 'Spring', 'Winter'],
    data=weekends_data,
    dodge=False,
    palette=fixed_season_palette
)
plt.title('Aluguel Diário de Bicicletas – Finais de Semana (Colorido por Estação)', fontsize=14, fontweight='bold')
plt.xlabel('Data', fontsize=12)
plt.ylabel('Total de Aluguéis', fontsize=12)
plt.xticks([])
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Objetivo: Exibir a série temporal diária completa diferenciando dias de semana e finais de semana
bikes_daily_daytype = df.groupby(['Date_Parsed', 'Day_Type'])['Rented_Bike_Count'].sum().reset_index()
bikes_daily_daytype = bikes_daily_daytype.sort_values(by='Date_Parsed')

plt.figure(figsize=(15, 6))
sns.barplot(
    x='Date_Parsed',
    y='Rented_Bike_Count',
    hue='Day_Type',
    hue_order=['Weekday', 'Weekend'],
    data=bikes_daily_daytype,
    dodge=False,
    palette='Set2'
)

plt.title('Total de Bicicletas Alugadas por Dia (Colorido por Dia de Semana vs. Final de Semana)', fontsize=14, fontweight='bold')
plt.xlabel('Data (Ordenada Cronologicamente)', fontsize=12)
plt.ylabel('Total de Bicicletas Alugadas', fontsize=12)
plt.xticks([])
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### O que observamos na análise de dias úteis e finais de semana

A decomposição temporal entre dias úteis e finais de semana mostra que:
- Dias de semana acumulam maior volume total porque representam 5 dos 7 dias da semana.
- Contudo, na série temporal cronológica, a trajetória de aluguéis em dias úteis e finais de semana é muito parecida, com quedas acentuadas ocorrendo simultaneamente em ambos.
- Isso indica que o tipo de dia não é a causa das quedas pontuais de aluguel. A explicação parece estar ligada a características específicas daquele dia (como eventos climáticos), e não ao calendário em si.

Antes de passarmos para as variáveis de clima, resta checar um último fator de calendário e operação: o funcionamento do sistema.


### 4.4 Status de Operação do Sistema (`Functioning_Day`)

Pergunta da análise: as datas com contagem de aluguéis próxima ou igual a zero decorrem de indisponibilidade operacional do próprio sistema?


In [ ]:
# Objetivo: Visualizar a série temporal de aluguéis destacando dias com sistema ativo versus inativo
bikes_daily_functioning = df.groupby(['Date_Parsed', 'Functioning_Day'])['Rented_Bike_Count'].sum().reset_index()
bikes_daily_functioning = bikes_daily_functioning.sort_values(by='Date_Parsed')

plt.figure(figsize=(15, 6))
sns.barplot(
    x='Date_Parsed',
    y='Rented_Bike_Count',
    hue='Functioning_Day',
    hue_order=['Yes', 'No'],
    data=bikes_daily_functioning,
    dodge=False,
    palette='Set2'
)

plt.title('Total Diário de Aluguéis (Colorido por Funcionamento do Sistema: Ativo vs. Inativo)', fontsize=14, fontweight='bold')
plt.xlabel('Data (Ordenada Cronologicamente)', fontsize=12)
plt.ylabel('Total de Bicicletas Alugadas', fontsize=12)
plt.xticks([])
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Objetivo: Inspecionar os registros onde o sistema esteve inativo (Functioning_Day == 'No')
dias_inativos = bikes_daily_functioning[bikes_daily_functioning["Functioning_Day"] == "No"]
print(f"Total de datas com registro de inatividade: {len(dias_inativos)}")
dias_inativos

### O que observamos e transição para as variáveis numéricas

A verificação do funcionamento do sistema trouxe uma resposta direta para parte dos dados:
- Em todos os dias em que o sistema esteve inativo (`Functioning_Day == 'No'`), a contagem de aluguéis é estritamente zero (`Rented_Bike_Count == 0`). Isso explica totalmente vários pontos nulos encontrados na série temporal.
- Com isso, o comportamento do calendário e da operação fica delimitado: o inverno reduz o volume geral de aluguéis, feriados têm demanda variável conforme a época do ano, dias de semana e fins de semana têm dinâmicas parecidas, e dias sem funcionamento têm aluguel nulo.
- A pergunta central que segue aberta é: o que faz a demanda despencar em dias normais de funcionamento (`Functioning_Day == 'Yes'`), inclusive durante as estações quentes?

A resposta provável reside nas variáveis numéricas e meteorológicas (chuva, neve, umidade elevada, vento e temperaturas extremas). A próxima etapa da análise exploratória será dedicada a investigar essas variáveis contínuas e sua relação com a demanda.


## Seção 5 – Análise Exploratória: Distribuição das Variáveis Numéricas e Outliers

Na seção anterior, vimos que o inverno reduz o volume geral de aluguéis e que dias sem funcionamento do sistema zeram a demanda, mas restou a pergunta: o que explica as quedas acentuadas de aluguéis em dias normais de funcionamento, inclusive durante as estações quentes?

Para responder a essa pergunta, precisamos analisar as variáveis contínuas e meteorológicas. Investigamos a distribuição de cada variável numérica para entender sua assimetria, concentração e identificar possíveis outliers.

As variáveis numéricas sob análise são:
- Variável resposta: `Rented_Bike_Count`
- Variáveis meteorológicas: `Temperature`, `Humidity`, `Wind_Speed`, `Visibility`, `Dew_Point_Temperature`, `Solar_Radiation`, `Rainfall`, `Snowfall`


### 5.1 Histogramas e Formato das Distribuições

Pergunta da análise: qual é o formato da distribuição das variáveis numéricas e quais apresentam assimetria ou concentração excessiva em zero?


In [ ]:
# Objetivo: Plotar histogramas com KDE para visualizar o formato da distribuição das variáveis numéricas
cols_numericas = [
    'Rented_Bike_Count', 'Temperature', 'Humidity', 'Wind_Speed',
    'Visibility', 'Dew_Point_Temperature', 'Solar_Radiation', 'Rainfall', 'Snowfall'
]

plt.figure(figsize=(16, 12))
for i, col in enumerate(cols_numericas, 1):
    plt.subplot(3, 3, i)
    sns.histplot(df[col], kde=True, color='#31688e', bins=30)
    plt.title(f'Distribuição de {col}', fontsize=12, fontweight='bold')
    plt.xlabel(col, fontsize=10)
    plt.ylabel('Frequência', fontsize=10)
    plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


### O que observei nos histogramas

Pelos gráficos de distribuição fico com a suspeita de que `Visibility`, `Solar_Radiation`, `Rainfall` e `Snowfall` tenham outliers. É claro que isso pode fazer parte do comportamento comum dos dados. Como o caso do `Snowfall`: em Seul, a neve acontece basicamente em um único período do ano (no inverno, de dezembro a fevereiro), com pico mais provável de fim de dezembro até o início de fevereiro, especialmente em janeiro.

Fonte de referência sobre o clima de Seul: https://english.visitseoul.net/essential-Info-article/Climate/ENN000385


### 5.2 Boxplots e Identificação de Outliers

Pergunta da análise: quais variáveis apresentam valores atípicos (outliers) pelo critério estatístico do IQR e qual é o significado prático desses valores?


In [ ]:
# Objetivo: Plotar boxplots para identificar a dispersão e presença de outliers nas variáveis numéricas
plt.figure(figsize=(16, 12))
for i, col in enumerate(cols_numericas, 1):
    plt.subplot(3, 3, i)
    sns.boxplot(y=df[col], color='#21918c', flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.5})
    plt.title(f'Boxplot de {col}', fontsize=12, fontweight='bold')
    plt.ylabel(col, fontsize=10)
    plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


### O que observei nos boxplots

A maioria dos gráficos aparenta ter um comportamento esperado, com exceção de `Solar_Radiation`, `Rainfall`, `Snowfall` e `Wind_Speed` que estão bem atípicos. Possivelmente eles estão diretamente correlacionados aos indicadores de chuva e neve. Seul fica em uma zona temperada com estações bem definidas: se só chover em um mês lá, qualquer indício de chuva vira outlier. Para isso vou investigar melhor esse comportamento dos dados com relação às estações.
